# Phase 3: Feature Engineering

## Imports 

In [0]:
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar

### Extract Data

In [0]:
hourly_demand = spark.sql("""
    SELECT * FROM hourly_demand
    WHERE hour >= '2019-02-01' AND hour < '2026-01-01'
""").toPandas()

#### Pivot

In [0]:
hourly_demand_pivoted = hourly_demand.pivot(index="hour", columns="vehicle_type", values="trip_count")
hourly_demand_pivoted.head()


vehicle_type,green,hvfhv,yellow
hour,,,
2019-02-01 00:00:00,410.0,23513.0,6994.0
2019-02-01 01:00:00,261.0,13638.0,4018.0
2019-02-01 02:00:00,155.0,8765.0,2413.0
2019-02-01 03:00:00,125.0,6668.0,1684.0
2019-02-01 04:00:00,133.0,7379.0,1654.0


### Creating New Cols

In [0]:
hourly_demand_pivoted['total_trips'] = hourly_demand_pivoted[['green', 'hvfhv', 'yellow']].sum(axis=1)

In [0]:
hourly_demand_pivoted['hour_of_day'] = hourly_demand_pivoted.index.hour
hourly_demand_pivoted['day_of_week'] = hourly_demand_pivoted.index.dayofweek
hourly_demand_pivoted['month'] = hourly_demand_pivoted.index.month
hourly_demand_pivoted['is_weekend'] = hourly_demand_pivoted['day_of_week'] >= 5


In [0]:
# cal is the rulebook / calander object that knows when the holiday dates fall
cal = USFederalHolidayCalendar()
# for this date range, which days/dates are holidays - datetimeindex / list of dates
holidays = cal.holidays(start=hourly_demand_pivoted.index.min(), end=hourly_demand_pivoted.index.max())
# nomalize strips the time component
hourly_demand_pivoted['is_holiday'] = hourly_demand_pivoted.index.normalize().isin(holidays)

hourly_demand_pivoted.head()

vehicle_type,green,hvfhv,yellow,total_trips,hour_of_day,day_of_week,month,is_weekend,is_holiday
hour,,,,,,,,,
2019-02-01 00:00:00,410.0,23513.0,6994.0,30917.0,0,4,2,False,False
2019-02-01 01:00:00,261.0,13638.0,4018.0,17917.0,1,4,2,False,False
2019-02-01 02:00:00,155.0,8765.0,2413.0,11333.0,2,4,2,False,False
2019-02-01 03:00:00,125.0,6668.0,1684.0,8477.0,3,4,2,False,False
2019-02-01 04:00:00,133.0,7379.0,1654.0,9166.0,4,4,2,False,False


In [0]:
for i in range(1, 25):
    hourly_demand_pivoted[f'lag_{i}h'] = hourly_demand_pivoted['total_trips'].shift(i)

hourly_demand_pivoted.head(24)

vehicle_type,green,hvfhv,yellow,total_trips,hour_of_day,day_of_week,month,is_weekend,is_holiday,lag_1h,lag_2h,lag_3h,lag_4h,lag_5h,lag_6h,lag_7h,lag_8h,lag_9h,lag_10h,lag_11h,lag_12h,lag_13h,lag_14h,lag_15h,lag_16h,lag_17h,lag_18h,lag_19h,lag_20h,lag_21h,lag_22h,lag_23h,lag_24h
hour,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2019-02-01 00:00:00,410.0,23513.0,6994.0,30917.0,0,4,2,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 01:00:00,261.0,13638.0,4018.0,17917.0,1,4,2,False,False,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 02:00:00,155.0,8765.0,2413.0,11333.0,2,4,2,False,False,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 03:00:00,125.0,6668.0,1684.0,8477.0,3,4,2,False,False,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 04:00:00,133.0,7379.0,1654.0,9166.0,4,4,2,False,False,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 05:00:00,177.0,11564.0,2902.0,14643.0,5,4,2,False,False,9166.0,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 06:00:00,513.0,22490.0,7582.0,30585.0,6,4,2,False,False,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 07:00:00,981.0,39850.0,13908.0,54739.0,7,4,2,False,False,30585.0,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 08:00:00,1542.0,51060.0,17090.0,69692.0,8,4,2,False,False,54739.0,30585.0,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [0]:
hourly_demand_pivoted['rolling_mean_24h'] = hourly_demand_pivoted['total_trips'].rolling(window=24).mean()
hourly_demand_pivoted['rolling_mean_168h'] = hourly_demand_pivoted['total_trips'].rolling(window=168).mean()
hourly_demand_pivoted.head(30)


vehicle_type,green,hvfhv,yellow,total_trips,hour_of_day,day_of_week,month,is_weekend,is_holiday,lag_1h,lag_2h,lag_3h,lag_4h,lag_5h,lag_6h,lag_7h,lag_8h,lag_9h,lag_10h,lag_11h,lag_12h,lag_13h,lag_14h,lag_15h,lag_16h,lag_17h,lag_18h,lag_19h,lag_20h,lag_21h,lag_22h,lag_23h,lag_24h,rolling_mean_24h,rolling_mean_168h
hour,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2019-02-01 00:00:00,410.0,23513.0,6994.0,30917.0,0,4,2,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 01:00:00,261.0,13638.0,4018.0,17917.0,1,4,2,False,False,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 02:00:00,155.0,8765.0,2413.0,11333.0,2,4,2,False,False,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 03:00:00,125.0,6668.0,1684.0,8477.0,3,4,2,False,False,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 04:00:00,133.0,7379.0,1654.0,9166.0,4,4,2,False,False,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 05:00:00,177.0,11564.0,2902.0,14643.0,5,4,2,False,False,9166.0,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 06:00:00,513.0,22490.0,7582.0,30585.0,6,4,2,False,False,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 07:00:00,981.0,39850.0,13908.0,54739.0,7,4,2,False,False,30585.0,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-02-01 08:00:00,1542.0,51060.0,17090.0,69692.0,8,4,2,False,False,54739.0,30585.0,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Test / Train Split Sanity Check

In [0]:
train = hourly_demand_pivoted[hourly_demand_pivoted.index < '2025-01-01']
test = hourly_demand_pivoted[hourly_demand_pivoted.index >= '2025-01-01']

print(len(train)) 
print(len(test))

51864
8760


### Save to Delta Table

In [0]:
# pandas -> pyspark
features_spark_df = spark.createDataFrame(hourly_demand_pivoted.reset_index())
# spark -> delta table
features_spark_df.write.format("delta").mode("overwrite").saveAsTable("features")

In [0]:
display(features_spark_df.limit(10))


hour,green,hvfhv,yellow,total_trips,hour_of_day,day_of_week,month,is_weekend,is_holiday,lag_1h,lag_2h,lag_3h,lag_4h,lag_5h,lag_6h,lag_7h,lag_8h,lag_9h,lag_10h,lag_11h,lag_12h,lag_13h,lag_14h,lag_15h,lag_16h,lag_17h,lag_18h,lag_19h,lag_20h,lag_21h,lag_22h,lag_23h,lag_24h,rolling_mean_24h,rolling_mean_168h
2019-02-01T00:00:00.000Z,410.0,23513.0,6994.0,30917.0,0,4,2,false,false,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T01:00:00.000Z,261.0,13638.0,4018.0,17917.0,1,4,2,false,false,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T02:00:00.000Z,155.0,8765.0,2413.0,11333.0,2,4,2,false,false,17917.0,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T03:00:00.000Z,125.0,6668.0,1684.0,8477.0,3,4,2,false,false,11333.0,17917.0,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T04:00:00.000Z,133.0,7379.0,1654.0,9166.0,4,4,2,false,false,8477.0,11333.0,17917.0,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T05:00:00.000Z,177.0,11564.0,2902.0,14643.0,5,4,2,false,false,9166.0,8477.0,11333.0,17917.0,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T06:00:00.000Z,513.0,22490.0,7582.0,30585.0,6,4,2,false,false,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T07:00:00.000Z,981.0,39850.0,13908.0,54739.0,7,4,2,false,false,30585.0,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T08:00:00.000Z,1542.0,51060.0,17090.0,69692.0,8,4,2,false,false,54739.0,30585.0,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2019-02-01T09:00:00.000Z,1618.0,43088.0,15584.0,60290.0,9,4,2,false,false,69692.0,54739.0,30585.0,14643.0,9166.0,8477.0,11333.0,17917.0,30917.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


### Check Row Counts

In [0]:
hourly_demand_pivoted.shape[0]

60624

In [0]:
features_spark_df.count()

60624